## LLM Setup

In [ ]:
!pip install -U transformers -q

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

LLM_ID = "Qwen/Qwen3.5-4B"

processor = AutoProcessor.from_pretrained(LLM_ID)
model = AutoModelForImageTextToText.from_pretrained(LLM_ID, device_map="auto")

## LLM Helper Functions

In [ ]:
def create_message(content, role="user"):
    '''
    This function lets you format text input into the format a LLM requires.
    
    role - The entity the message corresponds to. Should be one of 'user', 'helper', or 'system'.
    content - The actual content of the message being sent. Must be text.
    '''
    message = {
        "role": role, 
        "content": [
            {"type": "text", "text": content},
        ],
    }

    return message

In [ ]:
def generate_response(messages, max_new_tokens=1000):
    '''
    This function lets you input llm formatted messages and receive human readable llm output as 
    the result. Hyperparameter defaults are based on the recommendations for a conversational 
    qwen-3.5-4B model. 

    messages - list containing the message(s) llm will use as the basis for generating its output. 
    max_new_tokens - The max amount of generated tokens prior to the LLM cutting off its output.
    
    '''
    llm_input = processor.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)


    llm_output = model.generate(
        **llm_input, 
        max_new_tokens=max_new_tokens, 
        do_sample=True,
        temperature=.7, 
        top_p=0.8, 
        top_k=20, 
        min_p=0.0, 
        repetition_penalty=1.0,
    )

    readable_output = processor.decode(llm_output[0][llm_input["input_ids"].shape[-1]:], skip_special_tokens=True)

    return readable_output

## Conversation Helper Functions

In [ ]:
import random as random

def generate_personalized_system_prompt(base_system_prompt, trait_details):
    '''
    A function for generating a system prompt personalized to the trait of a specific user.
    Will randomly select one of the possible traits provided. 
    
    base_prompt - The prompt that describes the base purpose of the LLM. E.g. that it is a 
    shopping chatbot named ShoppingBot.
    
    trait_descriptions - A list of strings that correspond to the possible traits which a synthetic
    user may have.
    
    '''
    
    trait_selected = random.randint(0, len(trait_details)-1)
    trait_selected_details = trait_details[trait_selected]
    
    system_prompt = base_system_prompt + trait_selected_details
    system_prompt_message = create_message(system_prompt, "system")
    
    return trait_selected, trait_selected_details, system_prompt, system_prompt_message

In [ ]:
import pandas as pd
import random as rand
import copy

def generate_user_prompt(conversation, first_prompt=False):
    '''
    A function for generating synthetic user prompts. Or in other words, having an LLM 
    generate an example response which would be sent to another LLM.

    Generates prompts slightly differently based on a randomly decided relatedness to the previous
    llm conversation. Options are unrelated, somewhat related, and related.

    conversation - List with messages corresponding to the conversation so far. Includes the system 
    prompt, followed by a repeating sequence of previous synthetic user prompts followed by 
    the llm response to that user prompt.
    '''
    relatedness = rand.random()
    llm_input = []
    
    # Continue conversation with unrelated user prompt
    if relatedness < (1/3) or first_prompt == True:
        if first_prompt == True: 
            relation = "N/A"
        else:
            relation = "unrelated"
        llm_prompt = ["Give me an example of a realistic user prompt to this chatbot." 
                         "Output just that example and nothing else."]
        fake_conversation = [copy.deepcopy(conversation[0])]

    # Continue conversation with a somewhat related user prompt
    elif relatedness < (2/3): 
        relation = "somewhat"
        llm_prompt = ["Give me an example of a realistic user response to this that is somewhat"
                         "related to the pre-existing conversation. Output just that example and"
                         "nothing else."]
        fake_conversation = copy.deepcopy(conversation)
        
    # Continue conversation with a related user prompt
    else:
        relation = "related"
        llm_prompt = ["Give me a realistic user response to this that continues the pre-existing" 
                         "conversation. Output just that example and nothing else."]
        fake_conversation = copy.deepcopy(conversation)

    # Generating a synthetic user prompt based using the randomly selected relatedness.
    llm_prompt_message = create_message(llm_prompt, "user")
    fake_conversation.append(llm_prompt_message)
    new_user_prompt = generate_response(fake_conversation)
    
    return new_user_prompt, relation

In [ ]:
def generate_conversation(base_system_prompt, trait_descriptions, conversation_length=5):
    '''
        Generate a fully synthetic LLM conversation of the specified number of exchanges. Conversation
        centers around both the specified purpose of the llm (base_system_prompt) and a randomly selected
        trait corresponding to the synthetic "user" who is sending the llm questions. 

        base_system_prompt - The prompt that describes the base purpose of the LLM. E.g. that it is a 
        shopping chatbot named ShoppingBot.
        
        trait_descriptions - A list of strings that correspond to the possible traits which a synthetic
        user may have.
        
        conversation_length - length of the conversation being held in number of user input/llm output
        exchanges
    '''
    
    conversation = []
    user_prompts = []
    user_prompt_relatedness = []
    llm_responses = []

    # Generate the system prompt
    system_prompt_info = generate_personalized_system_prompt(base_system_prompt, trait_descriptions)
    trait_selected, trait_selected_details, system_prompt, system_prompt_message = system_prompt_info
    conversation.append(system_prompt_message)

    # Generate synthetic user prompts and llm responses for each conversation layer
    for i in range(conversation_length):

        # Generating synthetic user prompt
        user_prompt, relation = generate_user_prompt(conversation, i == 0)
        user_prompts.append(user_prompt)
        user_prompt_relatedness.append(relation) 
        conversation.append(create_message(user_prompt))

        # Generating llm response to synthetic user prompt.
        llm_response = generate_response(conversation)
        llm_responses.append(llm_response)
        conversation.append(create_message(llm_response, "assistant"))
    
    return (system_prompt, trait_selected, trait_selected_details, user_prompts, 
            user_prompt_relatedness, llm_responses)

## Dataset Generation Function

In [ ]:
from tqdm import tqdm
import os

def add_to_dataset(num_conversations, base_system_prompt, trait_descriptions, conversation_length,
                  dataset_location="synthetic_conversations.csv"):
    '''
    Adds LLM conversations corresponding to the specified function parameters to a csv dataset.

    num_conversations - Integer corresponding to the total number of conversations that should be 
    added to the dataset.
    
    base_system_prompt - String prompt that describes the base purpose of the LLM. E.g. that it is a 
    shopping chatbot named ShoppingBot.
    
    trait_descriptions - A list of strings that correspond to the possible traits which a synthetic
    user may have.
    
    conversation_length - Integer length of the conversation being held defined by number of responses 
    generated by the llm

    dataset_location - String location of the csv dataset being generated/added to.
    
    '''
    
    for i in tqdm(range(num_conversations)):
        data_id = rand.randint(0, 999999999)
        conversation_info = generate_conversation(base_system_prompt, trait_descriptions, 
                                                  conversation_length)
        (system_prompt, trait_selected, trait_selected_details, user_prompts, 
        user_prompt_relatedness, llm_responses) = conversation_info
        data = {
            'llm_id': LLM_ID,
            'data_id': data_id,
            'system_prompt': [system_prompt],
            'base_system_prompt': base_system_prompt, 
            'trait_selected': trait_selected,
            'trait_selected_details': trait_selected_details,            
            }
        for pos in range(conversation_depth):
            data[f'user_prompt_{pos}'] = user_prompts[pos]
            data[f'relatedness_{pos}'] = user_prompt_relatedness[pos]
            data[f'llm_output_{pos}'] = llm_responses[pos]
        
        df = pd.DataFrame(data)
        df.to_csv(dataset_location, mode='a', header=not os.path.exists(dataset_location))

## Generating the Dataset

In [ ]:
num_conversations = 200
base_system_prompt = "You are ShoppingBot. It is your purpose to help people make purchase decisions."
trait_descriptions = ["",
    " The user is very budget-conscious and prefers low-cost or highly affordable options.",
    " The user has moderate spending flexibility and values a balance between cost and quality.",
    " The user prioritizes premium experiences and is not sensitive to price."]
conversation_depth = 5

In [ ]:
add_to_dataset(num_conversations, base_system_prompt, trait_descriptions, conversation_depth)